# Mitigation Analysis

**Purpose:** Comprehensive analysis of insulation mitigation potential, cost-efficiency,
NPV assessment, MAC curves, and aggregated cost curves.

**Project imports:** `project.model` (get_inputs, social_planner), `project.utils` (cumulated_plot, cumulated_plots, make_plots),
`project.read_input` (generate_price_scenarios).

**Inputs:** Building stock, energy prices, insulation costs, carbon data, health costs.

**Outputs:** Mitigation potential tables, MAC curves, NPV curves, cost curves by stock variant.

**Consolidates:** Previously split across `mitigation_potential.ipynb`, `mac_curve.ipynb`,
`cost_efficiency_insulation.ipynb`, and `cost_curve.ipynb`.

## 1. Setup and data loading

In [ ]:
import os
import pandas as pd
from pandas import concat, Series, DataFrame
from numpy import inf
import matplotlib.pyplot as plt

from project.model import get_inputs, social_planner
from project.utils import cumulated_plot, cumulated_plots, make_plots

In [ ]:
path = 'output/static'
if not os.path.isdir(path):
    os.mkdir(path)

In [ ]:
insulation = {
    'Wall': (True, False, False, False),
    'Floor': (False, True, False, False),
    'Roof': (False, False, True, False),
    'Windows': (False, False, False, True),
    'Global': (True, True, True, True)
}

insulation_index = pd.MultiIndex.from_frame(
    pd.DataFrame({k: v for k, v in insulation.items() if k != 'Global'})
)

### Loading inputs

In [ ]:
output = get_inputs(path)
buildings = output['buildings']
energy_prices = output['energy_prices']
cost_insulation = output['cost_insulation']
carbon_emission = output['carbon_emission']
carbon_value_kwh = output['carbon_value_kwh']
health_cost = output['health_cost']

## 2. Mitigation potential

Calculating:
1. Initial consumption of the building stock
2. Potential energy reduction after insulation measures
3. Cost of insulation measures
4. Cost-efficiency of potential measures
5. Marginal Abatement Cost (MAC) curve

In [ ]:
output = buildings.mitigation_potential(energy_prices, cost_insulation, carbon_emission, carbon_value_kwh)
output.keys()

### Actual consumption of the building stock

In [ ]:
total_surface = (buildings.surface * buildings.stock).sum()
print('{:,.0f} Million m2'.format(total_surface / 10**6))

consumption_total_before = output['Consumption actual before (kWh/segment)'].sum()
print('Actual space heating: {:,.0f} TWh'.format(consumption_total_before / 10**9))

consumption_standard_before = output['Consumption before (kWh/segment)'].sum()
print('Conventional space heating: {:,.0f} TWh'.format(consumption_standard_before / 10**9))

#### Thermal transmittance distribution

In [ ]:
temp = dict()
for i in ['Wall', 'Floor', 'Roof', 'Windows']:
    y = Series(buildings.stock.index.get_level_values(i), index=buildings.stock.index,
              name='{} insulation (W/m2.K)'.format(i)).astype('float')
    x = buildings.stock / 10**6
    temp.update({i: cumulated_plot(x, y, plot=False)})
cumulated_plots(temp, 'Thermal transmittance U (W/m2.K)', ylim=3)

#### kWh/m2 distribution

In [ ]:
print('Average actual: {:.0f} kWh/m2'.format(
    output['Consumption actual before (kWh/segment)'].sum() / total_surface))
print('Average conventional: {:.0f} kWh/m2'.format(
    output['Consumption before (kWh/segment)'].sum() / total_surface))

(output['Consumption actual before (kWh/dwelling)'] / buildings.surface).describe()

In [ ]:
print(output['Consumption actual before (kWh/dwelling)'].describe())
output['Consumption actual before (kWh/dwelling)'].rename('Consumption (kWh/dwelling)').plot.box()

### Potential energy consumption saved by insulation measures

#### Final energy consumption after insulation

In [ ]:
consumption_total_after = output['Consumption actual after (kWh/segment)'].sum()
concat((consumption_total_after / 10**9, consumption_total_after / total_surface),
       keys=['Consumption (TWh)', 'Consumption (kWh/m2)'], axis=1)

#### Energy saving

In [ ]:
consumption_total_saved = output['Consumption actual saved (kWh/segment)'].sum().loc[insulation.values()]
concat((consumption_total_saved / 10**9,
        consumption_total_saved / total_surface,
        consumption_total_saved / consumption_total_before),
       keys=['Consumption saved (TWh/year)', 'Consumption saved (kWh/m2)', 'Percentage saved (%)'],
       axis=1)

Roof and floor contribute less than expected — they only target single-family dwellings.

In [ ]:
output['Consumption actual saved (kWh/dwelling)'].loc[:, insulation.values()].describe()

### Cost of measures

In [ ]:
output['Cost insulation (euro/dwelling)'].loc[:, insulation.values()].describe()

In [ ]:
cost = (output['Cost insulation (euro/dwelling)'].T / buildings.surface).T
cost.loc[:, insulation.values()].describe()

## 3. Cost-efficiency of renovation measures

In [ ]:
discount_rate = 0.04
discount_factor = (1 - (1 + discount_rate) ** -30) / discount_rate
print('Discount factor (4%, 30y):', round(discount_factor, 2))

In [ ]:
consumption_saved = output['Consumption saved (kWh/segment)']
cost_efficiency = output['Cost insulation (euro/dwelling)'] / (output['Consumption saved (kWh/dwelling)'] * discount_factor)
cost_efficiency.replace([inf, -inf], 0, inplace=True)
cost_efficiency.fillna(0, inplace=True)

In [ ]:
temp, temp_percent, temp_stock = dict(), dict(), dict()
for key, item in insulation.items():
    x = consumption_saved.loc[:, item].rename('Consumption saved (TWh)') / 10**9
    x_percent = (consumption_saved.loc[:, item] / consumption_standard_before).rename('Consumption saved (%)')
    x_stock = buildings.stock / 10**6

    y = cost_efficiency.loc[:, item].rename('{} insulation'.format(key))

    temp.update({key: cumulated_plot(x, y, plot=False)})
    temp_percent.update({key: cumulated_plot(x_percent, y, plot=False)})
    temp_stock.update({key: cumulated_plot(x_stock, y, plot=False)})

In [ ]:
cumulated_plots(temp, 'Cost (€/kWh)')
cumulated_plots(temp_percent, 'Cost (€/kWh)')
cumulated_plots(temp_stock, 'Cost (€/kWh)')

## 4. Net Present Value assessment

In [ ]:
discount_rate = 0.08
discount_factor = (1 - (1 + discount_rate) ** -30) / discount_rate
print('Discount factor (8%, 30y):', round(discount_factor, 2))

In [ ]:
npv = output['Cost insulation (euro/dwelling)'] - output['Bill saved (euro/dwelling)'] * discount_factor

In [ ]:
i = 'Floor'
details = concat(
    (output['Cost insulation (euro/dwelling)'].loc[:, insulation[i]],
     output['Bill saved (euro/dwelling)'].loc[:, insulation[i]] * discount_factor,
     npv.loc[:, insulation[i]].rename('NPV (euro/dwelling)')),
    axis=1
).sort_values('NPV (euro/dwelling)')

In [ ]:
temp, temp_percent, temp_stock = dict(), dict(), dict()
for key, item in insulation.items():
    x = consumption_saved.loc[:, item].rename('Consumption saved (TWh)') / 10**9
    x_percent = (consumption_saved.loc[:, item] / consumption_standard_before).rename('Consumption saved (%)')
    x_stock = buildings.stock / 10**6

    y = npv.loc[:, item].rename('{} insulation'.format(key)) / 10**3

    temp.update({key: cumulated_plot(x, y, plot=False)})
    temp_percent.update({key: cumulated_plot(x_percent, y, plot=False)})
    temp_stock.update({key: cumulated_plot(x_stock, y, plot=False)})

cumulated_plots(temp, 'NPV (Thousand euro/dwelling)', ymin=None)
cumulated_plots(temp_percent, 'NPV (Thousand euro/dwelling)', ymin=None)
cumulated_plots(temp_stock, 'NPV (Thousand euro/dwelling)', ymin=None)

### Social NPV assessment
Includes carbon emission value in addition to private bill savings.

In [ ]:
social_npv = output['Cost insulation (euro/dwelling)'] - (
    output['Bill saved (euro/dwelling)'] + output['Emission value saved (euro/dwelling)']
) * discount_factor

In [ ]:
temp, temp_percent, temp_stock = dict(), dict(), dict()
for key, item in insulation.items():
    x = consumption_saved.loc[:, item].rename('Consumption saved (TWh)') / 10**9
    x_percent = (consumption_saved.loc[:, item] / consumption_standard_before).rename('Consumption saved (%)')
    x_stock = buildings.stock / 10**6

    y = social_npv.loc[:, item].rename('{} insulation'.format(key)) / 10**3

    temp.update({key: cumulated_plot(x, y, plot=False)})
    temp_percent.update({key: cumulated_plot(x_percent, y, plot=False)})
    temp_stock.update({key: cumulated_plot(x_stock, y, plot=False)})

cumulated_plots(temp, 'Social NPV (Thousand euro/dwelling)', ymin=None)
cumulated_plots(temp_percent, 'Social NPV (Thousand euro/dwelling)', ymin=None)
cumulated_plots(temp_stock, 'Social NPV (Thousand euro/dwelling)', ymin=None)

## 5. MAC curves — Best NPV vs Max consumption saved

### Marginal Energy Saving Cost Curves (MESC-Curves)
These curves show for each amount of energy demand savings (x-axis) the costs of the last
renovation action to be taken to reach these savings (y-axis).

### Energy Saving Cost Curves (ESC-Curves)
These curves show for each amount of energy demand savings (x-axis) the average costs of all
renovation actions to be implemented to reach these savings (y-axis).

In [ ]:
discount_rate, lifetime = 0.05, 30
discount_factor = (1 - (1 + discount_rate) ** -lifetime) / discount_rate
print('Discount factor (5%, 30y):', round(discount_factor, 2))

In [ ]:
result = dict()

for scenario in ['Best NPV', 'Max consumption saved']:
    if scenario not in output:
        continue
    df = output[scenario].copy()
    df.drop(df[df['bill_saved'] == 0].index, inplace=True)
    df['consumption_saved_sum'] = df['consumption_saved'] * discount_factor
    df['consumption_saved_agg_sum'] = df['consumption_saved_agg'] * discount_factor
    df['cost_consumption'] = df['cost'] / df['consumption_saved_sum']

    x = (df['consumption_saved_agg'] / 10**9).rename('Consumption saved (TWh/a)')
    y = df['cost_consumption'].rename('Cost insulation (euro/kWh)')
    cumulated_plot(x, y, plot=True, format_y=lambda y, _: '{:,.1f}'.format(y))
    plt.title(scenario)
    result.update({scenario: cumulated_plot(x, y, plot=False)})

    print(f'{scenario}: Consumption conventional initial: '
          f'{output["Consumption before (kWh/segment)"].sum() / 10**9:,.0f} TWh/a')

if len(result) > 1:
    cumulated_plots(result, 'Cost insulation (euro/kWh)')

## 6. Parameterized cost-efficiency analysis

Reusable function to run cost-efficiency analysis on different building stocks and subsets.
Pass `building_stock` name and optionally filter by `level`/`val` (e.g., Housing type).

In [ ]:
def analysis_cost_efficiency(building_stock='simple_1', val=None, level=None):
    """Run cost-efficiency analysis for a given building stock variant.

    Parameters
    ----------
    building_stock : str
        Name of the stock CSV (e.g. 'simple_1', 'medium_5').
    val : str, optional
        Value to filter on (e.g. 'Single-family').
    level : str, optional
        Index level to filter (e.g. 'Housing type').
    """
    resirf_inputs = get_inputs(
        variables=['buildings', 'energy_prices', 'cost_insulation'],
        building_stock=os.path.join('project', 'input', 'stock',
                                    'buildingstock_sdes2018_{}.csv'.format(building_stock))
    )
    b = resirf_inputs['buildings']
    ep = resirf_inputs['energy_prices']
    ci = resirf_inputs['cost_insulation']

    out = b.mitigation_potential(ep, ci)

    cons_saved = out['Consumption saved (kWh/dwelling)'].loc[:, insulation_index]
    cost_ins = out['Cost insulation (euro/dwelling)'].loc[:, insulation_index]
    cost_ins[cons_saved == 0] = 0
    cons_before = out['Consumption before (kWh/dwelling)']
    ce = cost_ins / cons_saved

    stock = b.stock
    if level is not None:
        stock = stock.xs(val, level=level)
        cons_saved = cons_saved.xs(val, level=level)
        cons_before = cons_before.xs(val, level=level)
        ce = ce.xs(val, level=level)

    insulation_dict = {k: v for k, v in insulation.items() if k != 'Global'}
    result = dict()
    for n, i in insulation_dict.items():
        y = ce.loc[:, i].rename('Cost efficiency (euro/kWh)')
        x = cons_saved.loc[:, i].rename('Consumption saved (%)') / cons_before.sum()
        df = pd.concat((x, y), axis=1)
        df.sort_values(y.name, inplace=True)
        df['{} cumulated'.format(x.name)] = df[x.name].cumsum()
        df.dropna(inplace=True)
        df = df.set_index('{} cumulated'.format(x.name))[y.name]
        result.update({n: df})

    title = f'{building_stock}'
    if val:
        title += f' — {val}'
    print(title)
    make_plots(result, 'Cost efficiency (euro/kWh)')

    temp = dict()
    for i in ['Wall', 'Floor', 'Roof', 'Windows']:
        y = pd.Series(stock.index.get_level_values(i), index=stock.index,
                      name='{} insulation (W/m2.K)'.format(i)).astype('float')
        x = stock / 10**6
        temp.update({i: cumulated_plot(x, y, plot=False)})
    cumulated_plots(temp, 'Thermal transmittance U (W/m2.K)', ylim=3)

### Example: Single-family vs Multi-family with medium_5 stock

In [ ]:
analysis_cost_efficiency(building_stock='medium_5', val='Single-family', level='Housing type')
analysis_cost_efficiency(building_stock='medium_5', val='Multi-family', level='Housing type')

### Sweep across all stock variants

In [ ]:
for building_stock in ['simple_1', 'simple_3', 'simple_5', 'medium_1', 'medium_3', 'medium_5']:
    analysis_cost_efficiency(building_stock=building_stock, val='Single-family', level='Housing type')

## 7. Aggregated cost curves (social planner)

Uses the `social_planner()` function to create cost curves aggregated by different dimensions
(Housing type, Performance, Energy) and across different building stock variants.

In [ ]:
dict_cost, dict_heat = social_planner(aggregation_archetype=['Housing type'],
                                      building_stock='medium_3', freq='hour', percent=False)
make_plots(dict_cost, 'Cost by Energy Performance (Billion euro)', loc='left', left=1.15)

In [ ]:
dict_cost, dict_heat = social_planner(aggregation_archetype=['Housing type'],
                                      building_stock='medium_3', freq='hour', percent=True)
make_plots(dict_cost, 'Cost by Energy Performance (Billion euro)', loc='left', left=1.15)

In [ ]:
dict_cost, dict_heat = social_planner(aggregation_archetype=['Performance'],
                                      building_stock='medium_5', freq='hour', percent=False)
make_plots(dict_cost, 'Cost by Energy Performance (Billion euro)', loc='left', left=1.15)

In [ ]:
dict_cost, dict_heat = social_planner(aggregation_archetype=['Performance'],
                                      building_stock='medium_5')
make_plots(dict_cost, 'Cost by Energy Performance (Billion euro)', loc='left', left=1.15)

### By Energy type

In [ ]:
dict_cost, _ = social_planner(aggregation_archetype=['Energy'])
make_plots(dict_cost, 'Cost by Energy Performance (Billion euro)', loc='left', left=1.15)

### By Housing type

In [ ]:
dict_cost, _ = social_planner(aggregation_archetype=['Housing type'])
make_plots(dict_cost, 'Cost by Housing type (Billion euro)', loc='left', left=1.15)

### By Housing type x Performance

In [ ]:
dict_cost, dict_heat = social_planner(aggregation_archetype=['Housing type', 'Performance'])
make_plots({k: item for k, item in dict_cost.items() if 'Single-family' in k},
           'Cost (Billion euro)', loc='left', left=1.15)
make_plots({k: item for k, item in dict_cost.items() if 'Multi-family' in k},
           'Cost (Billion euro)', loc='left', left=1.15)

### Multi-stock comparison

In [ ]:
stocks = (['medium_{}'.format(i) for i in [1, 3, 5]]
          + ['simple_{}'.format(i) for i in [1, 3, 5]])

for stock_name in stocks:
    print(f'--- {stock_name} ---')
    dict_cost, _ = social_planner(
        aggregation_archetype=['Housing type'],
        building_stock=stock_name
    )
    make_plots(dict_cost, f'Cost — {stock_name} (Billion euro)', loc='left', left=1.15)